# Train π0.5 (`pi05`) on RunPod — FR5 pick-and-place — **self-contained**

π0.5 shares π0's architecture (PaliGemma-2B VLM + 300M action expert) but uses tokenizer_max_length=200 instead of 48 — a much richer language context window.

This notebook is **fully standalone**: no git clone, no repo files. The dataset class,
the π0.5 wrapper, and the training loop are all defined in cells below (ported from the
`fairino-fr5-policies` repo so **checkpoints stay byte-compatible with the repo's
`common/deploy.py`** on the robot box).

**Finetuning recipe:** pretrained `lerobot/pi05_base` weights (verified load with a hard
key-match check) + **LoRA adapters on the frozen VLM** + **full training of the 300M
action expert & projections**.

## Pod setup
| | |
|---|---|
| **Template** | RunPod **PyTorch 2.x** (CUDA ≥ 12.1) |
| **GPU** | 48 GB (A40 / A6000 / L40S) recommended · 24 GB (4090) works with `expert_only` · 80 GB = full finetune |
| **Disk** | ≥ 60 GB (~5 GB PaliGemma + dataset + checkpoints, ~5 GB each) |

## One-time prerequisites
1. **HF token** (read + write) whose account has **accepted the PaliGemma license**:
   <https://huggingface.co/google/paligemma-3b-pt-224> — gated; the tokenizer comes from there.
   The model weights themselves come from **`lerobot/pi05_base`** (~6 GB, the openpi port on the
   lerobot hub) — this is what makes it *finetuning*; without it lerobot random-inits the
   whole model.
2. **Convert raw episodes → LeRobot, then push to the Hub.** Your recordings are raw
   `episode_XXXX/` folders (data.csv + wrist_cam.mp4 + scene_cam.mp4 + meta.json), e.g. on
   the Hub as `Slifold/episodes_20260717`. **Easiest: run
   `notebooks/convert_and_push_dataset.ipynb`** — it pulls the raw set, converts, and pushes
   the LeRobot dataset; then set `HF_DATASET_REPO` below to that output repo. Manual path:
   First convert them with the repo's converter (produces a 7-D joint state + 7-D action,
   wrist-cam only, 30 fps, and a default task string when meta.json's instruction is empty):
   ```bash
   python common/convert_episodes.py --episodes episodes --out lerobot_dataset --extract-frames
   ```
   Then push the resulting `lerobot_dataset/` to the Hub:
   ```bash
   pip install huggingface_hub && huggingface-cli login
   python -c "from huggingface_hub import HfApi; api=HfApi(); \
       api.create_repo('<you>/fr5-pick-place-lerobot', repo_type='dataset', private=True, exist_ok=True); \
       api.upload_folder(folder_path='lerobot_dataset', repo_id='<you>/fr5-pick-place-lerobot', repo_type='dataset')"
   ```
   (or `python tools/push_dataset_hf.py --repo <you>/fr5-pick-place-lerobot` if you have the repo).

## 1 · Parameters

| variable | meaning |
|---|---|
| `FINETUNE_MODE` | `auto` picks by VRAM · **`lora`** = LoRA adapters on the frozen VLM **+ full action-expert training** (the recommended recipe) · `full` = finetune everything · `freeze_vision` = freeze SigLIP only · `expert_only` = freeze the whole VLM, no adapters |
| `BATCH_SIZE` | `None` = auto by VRAM |
| `PROPRIO_MODE` | `full` / `dropout` / `none` — proprioception benchmark axis |

**VRAM cheat-sheet** (bf16 + gradient checkpointing always on): 24 GB → `expert_only`, batch 2 · 48 GB → `lora`, batch 4 · 80 GB → `lora` batch 8 (or `full`).

> `lora` adapts the VLM to FR5 visuals with only ~7M extra trainable params while the
> 300M action expert trains fully — the best fit for 150 episodes. `expert_only` is the
> low-VRAM fallback (VLM completely untouched).

In [ ]:
import os

HF_TOKEN        = os.environ.get("HF_TOKEN", "")        # hf_... (read+write, PaliGemma licence accepted)
HF_DATASET_REPO = "<you>/fr5-pick-place-lerobot"

USE_WANDB     = True                                     # auto-disables if no API key
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")      # wandb.ai/authorize
WANDB_PROJECT = "fr5-vla-benchmark"

POLICY       = "pi05"
PRETRAINED   = "lerobot/pi05_base"  # "" -> random init (smoke tests only — NOT finetuning)
TASK_TEXT    = "pick up the block and place it in the bin"
FINETUNE_MODE = "auto"      # auto | lora | full | freeze_vision | expert_only
LORA_RANK    = 16           # LoRA rank on the VLM q/k/v/o (lora mode)
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
BATCH_SIZE   = None         # None -> picked from GPU VRAM
MAX_EPOCHS   = 100
LR           = 2.5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
CHUNK_SIZE   = 50           # action horizon (pi0 default, 1.67 s @ 30 Hz)
VAL_FRAC     = 0.1
AUG_LEVEL    = "crops"      # none | crops   (train-time image augmentation)
PROPRIO_MODE = "full"       # full | dropout | none
SAVE_EVERY   = 10           # epochs between periodic checkpoints
SEED         = 42

WORK     = "/workspace"
DATA_DIR = f"{WORK}/lerobot_dataset"
CKPT_DIR = f"{WORK}/checkpoints_{POLICY}"

# never hardcode the token. Preferred: set it as a pod secret / env var
#   (RunPod: Pod -> Edit -> Environment Variables -> HF_TOKEN = hf_...).
# Fallback: if it isn't in the environment, prompt for it here (masked input,
# not saved into the notebook).
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "no HF token — PaliGemma + the pretrained weights are gated"
print("parameters set")

## 2 · Install dependencies

`lerobot==0.5.1` pins the torch/transformers stack — ~3–5 min on first run.
(pip may replace the pod's preinstalled torch; that's expected.)

In [ ]:
import subprocess, sys

# everything the notebook needs, explicitly (lerobot pulls torch/torchvision/
# transformers/huggingface_hub/safetensors/opencv-headless as pinned deps).
# NOT quiet: pip output streams live so you can watch the download and, more
# importantly, SEE any dependency-resolution conflicts instead of a late crash.
subprocess.run([sys.executable, "-m", "pip", "install",
                "lerobot==0.5.1",        # PI policy implementation + pinned DL stack
                "peft>=0.17",            # LoRA adapters on the VLM
                "wandb>=0.17",           # experiment tracking
                "numpy>=2.0,<2.3",
                "pandas", "pyarrow",     # dataset parquet handling
                "matplotlib", "tqdm"], check=True)

import torch, lerobot, transformers, peft, wandb
print(f"\ntorch {torch.__version__} · lerobot {lerobot.__version__} · "
      f"transformers {transformers.__version__} · peft {peft.__version__} · "
      f"wandb {wandb.__version__}")

## 3 · HuggingFace auth + gated-weights check

Fails fast with the license URL if the token can't access PaliGemma.

In [ ]:
from huggingface_hub import login, whoami, auth_check
from huggingface_hub.errors import GatedRepoError

login(token=HF_TOKEN, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

try:
    auth_check("google/paligemma-3b-pt-224")
    print("PaliGemma licence OK — gated weights accessible")
except GatedRepoError:
    raise SystemExit("PaliGemma is gated for this token — accept the licence at "
                     "https://huggingface.co/google/paligemma-3b-pt-224 and re-run")

### 3b · Weights & Biases login

Optional but recommended — live loss curves, run comparison across policies /
finetune modes, and the GT-vs-prediction figure land in one dashboard.
No key → the notebook silently falls back to local `metrics.csv` + matplotlib only.

In [ ]:
import wandb

if USE_WANDB and WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("wandb: logged in — project", WANDB_PROJECT)
elif USE_WANDB:
    USE_WANDB = False
    print("wandb: no WANDB_API_KEY set -> disabled (metrics.csv still written)")
else:
    print("wandb: disabled")

## 4 · GPU check → resolve memory mode + batch size

In [ ]:
import torch

assert torch.cuda.is_available(), "no CUDA GPU — pick a GPU pod"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
assert torch.cuda.is_bf16_supported(), "bf16 unsupported — use an Ampere or newer GPU"
print(f"{name}  {vram:.0f} GB")

if FINETUNE_MODE == "auto":
    FINETUNE_MODE = "expert_only" if vram < 40 else "lora"
if BATCH_SIZE is None:
    BATCH_SIZE = {"expert_only": 2, "lora": 8 if vram >= 70 else 4,
                  "freeze_vision": 4, "full": 4 if vram >= 70 else 2}[FINETUNE_MODE]
print(f"finetune_mode = {FINETUNE_MODE}   batch_size = {BATCH_SIZE}")

## 5 · Pull the dataset from the Hub

In [ ]:
from huggingface_hub import snapshot_download
import json, pathlib

snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_DIR)
info = json.loads(pathlib.Path(DATA_DIR, "meta", "info.json").read_text())

# derive dims from the dataset itself — do NOT hardcode. The FR5 converter builds
# observation.state = 6 joints + gripper_norm (7) and action = 6 cmd joints +
# gripper (7); an older converter emitted a 6-D state, so read it to be safe.
STATE_DIM  = info["features"]["observation.state"]["shape"][0]
ACTION_DIM = info["features"]["action"]["shape"][0]
CAMERAS    = [k for k in info["features"] if k.startswith("observation.images.")]
print(f"episodes = {info['total_episodes']}   frames = {info['total_frames']}   "
      f"fps = {info['fps']}   robot = {info.get('robot_type')}")
print(f"state_dim = {STATE_DIM}   action_dim = {ACTION_DIM}   cameras = {CAMERAS}")
assert "observation.images.wrist_cam" in CAMERAS, \
    f"pi0/pi05 here use the wrist cam; dataset has {CAMERAS}"
if len(CAMERAS) > 1:
    print(f"note: {len(CAMERAS)} cameras present; this single-cam pipeline uses wrist_cam only")

### 5b · Eyeball the data

A wrist-cam frame and one episode's 7-D action traces — if these look wrong
(black frames, flat traces), stop and fix the dataset before spending GPU-hours.

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

df_raw = pd.read_parquet(pathlib.Path(DATA_DIR, "data/chunk-000/file-000.parquet"))
ep = df_raw[df_raw.episode_index == 0]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
frames = sorted(pathlib.Path(DATA_DIR, "frames").rglob("ep-000/*.jpg"))
if frames:
    ax[0].imshow(mpimg.imread(frames[len(frames) // 2])); ax[0].axis("off")
    ax[0].set_title(f"wrist cam, mid-episode ({len(frames)} frames)")
pd.DataFrame(ep["action"].tolist()).plot(ax=ax[1], legend=False,
    title=f"episode 0 action traces (7-D, {len(ep)} steps)")
plt.tight_layout(); plt.show()

## 6 · Dataset class (inline)

Ported from the repo's `common/dataset.py`: chunked action targets with padding,
episode-level train/val split (no leakage), ImageNet-normalised 224×224 frames,
and optional UMI-style crop/colour-jitter augmentation for training.

In [ ]:
import json, random
from pathlib import Path

import cv2
import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset
from torchvision import transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]


def _build_transform(image_size, aug_level):
    h, w = image_size
    if aug_level == "none":
        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    if aug_level == "crops":   # UMI-calibrated jitter
        return transforms.Compose([
            transforms.Resize((int(h * 1.12), int(w * 1.12))),
            transforms.RandomCrop(image_size),
            transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.5, hue=0.08),
            transforms.RandomGrayscale(p=0.05),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    raise ValueError(f"unknown aug_level {aug_level!r}")


class FR5Dataset(Dataset):
    """LeRobot-v3 FR5 episodes -> (state, action chunk, pad mask, image, task)."""

    def __init__(self, root, chunk_size=50, image_size=(224, 224),
                 episode_indices=None, aug_level="none"):
        self.root, self.chunk_size, self.image_size = Path(root), chunk_size, image_size
        self.info = json.loads((self.root / "meta/info.json").read_text())
        self.camera_keys = [k for k in self.info.get("features", {})
                            if k.startswith("observation.images.")] or                            ["observation.images.wrist_cam"]
        self.df = pq.read_table(self.root / "data/chunk-000/file-000.parquet").to_pandas()
        self.episodes = pq.read_table(
            self.root / "meta/episodes/chunk-000/file-000.parquet").to_pandas()
        if episode_indices is not None:
            self.episodes = self.episodes[
                self.episodes["episode_index"].isin(episode_indices)].reset_index(drop=True)
        self._samples = [(int(e.episode_index), t)
                         for _, e in self.episodes.iterrows()
                         for t in range(int(e.dataset_from_index), int(e.dataset_to_index))]
        tasks = self.root / "meta/tasks.parquet"
        self._task_map = (dict(zip(*pq.read_table(tasks).to_pandas()
                                   [["task_index", "task"]].T.values.tolist()))
                          if tasks.exists() else {})
        self._tf = _build_transform(image_size, aug_level)

    def __len__(self): return len(self._samples)

    def __getitem__(self, idx):
        ep_idx, frame_abs = self._samples[idx]
        row = self.df.iloc[frame_abs]
        ep = self.episodes[self.episodes.episode_index == ep_idx].iloc[0]
        ep_to = int(ep.dataset_to_index)

        state = torch.tensor(row["observation.state"], dtype=torch.float32)
        chunk = self.df.iloc[frame_abs:min(frame_abs + self.chunk_size, ep_to)]
        actions = torch.tensor(np.array(chunk["action"].tolist()), dtype=torch.float32)
        pad = self.chunk_size - len(actions)
        is_pad = torch.zeros(self.chunk_size, dtype=torch.bool)
        if pad > 0:
            actions = torch.cat([actions, actions[-1:].expand(pad, -1)])
            is_pad[-pad:] = True

        sample = {"observation.state": state, "action": actions, "action_is_pad": is_pad,
                  "task": self._task_map.get(int(row.get("task_index", 0)), TASK_TEXT)}
        for cam in self.camera_keys:                   # load EVERY camera (wrist + scene)
            jpg = self.root / "frames" / cam / f"ep-{ep_idx:03d}" / f"{int(row.frame_index):06d}.jpg"
            frame = cv2.imread(str(jpg))
            if frame is None:                          # video fallback
                cap = cv2.VideoCapture(str(self.root / "videos" / cam / "chunk-000" /
                                           f"file-{ep_idx:03d}.mp4"))
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame_index))
                ok, frame = cap.read(); cap.release()
                assert ok, f"missing {cam} frame ep{ep_idx} idx{int(row.frame_index)}"
            img = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                                   ).permute(2, 0, 1).float() / 255.0
            sample[cam] = self._tf(img)
        return sample

    def get_stats(self):
        sub = self.df[self.df.episode_index.isin(self.episodes.episode_index.tolist())]
        s, a = (np.array(sub[k].tolist()) for k in ("observation.state", "action"))
        return {"state_mean": s.mean(0).astype(np.float32),
                "state_std":  s.std(0).clip(1e-6).astype(np.float32),
                "state_min":  s.min(0).astype(np.float32),
                "state_max":  s.max(0).astype(np.float32),
                "action_mean": a.mean(0).astype(np.float32),
                "action_std":  a.std(0).clip(1e-6).astype(np.float32),
                "action_min":  a.min(0).astype(np.float32),
                "action_max":  a.max(0).astype(np.float32)}

    @staticmethod
    def episode_split(n_episodes, val_frac=0.1, seed=42):
        idx = list(range(n_episodes))
        random.seed(seed); random.shuffle(idx)
        n_val = max(1, int(val_frac * n_episodes))
        return idx[n_val:], idx[:n_val]

print("FR5Dataset defined")

## 7 · π0.5 policy wrapper (inline)

Ported from the repo's `policies/pi05/model.py` — identical normalisation, batch
building, proprio masking, and config threading, wrapping lerobot's `PI05Policy`.
The checkpoint this produces loads directly in the repo's `common/deploy.py`.

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
from transformers import AutoTokenizer

from lerobot.policies.pi05.configuration_pi05 import PI05Config as _LRConfig
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

STATE_KEY, IMAGE_KEY, ACTION_KEY = ("observation.state",
                                    "observation.images.wrist_cam", "action")
LANG_TOKENS    = "observation.language.tokens"
LANG_ATTN_MASK = "observation.language.attention_mask"
_IMN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMN_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def mask_state(state, mode, rate, training):
    """Proprio modes: full = untouched · none = always zeroed ·
    dropout = per-sample zeroed with prob `rate` during training only."""
    if mode == "none":
        return torch.zeros_like(state)
    if mode == "dropout" and training:
        keep = (torch.rand(state.shape[0], 1, device=state.device) >= rate)
        return state * keep.to(state.dtype)
    return state


@dataclass
class PolicyCfg:
    state_dim:  int = 7
    action_dim: int = 7
    chunk_size: int = 50
    use_image:  bool = True
    num_inference_steps: int = 10
    max_state_dim:  int = 32
    max_action_dim: int = 32
    paligemma_variant:     str = "gemma_2b"
    action_expert_variant: str = "gemma_300m"
    tokenizer_max_length:  int = 200
    pretrained:            str = "lerobot/pi05_base"   # "" -> random init (smoke only)
    vlm_lora_rank:         int = 0                  # >0 -> LoRA on VLM q/k/v/o
    vlm_lora_alpha:        int = 32
    vlm_lora_dropout:      float = 0.05
    dtype:                  str  = "bfloat16"
    gradient_checkpointing: bool = True
    freeze_vision_encoder:  bool = False
    train_expert_only:      bool = False
    camera_names:         tuple = ("wrist_cam", "scene_cam")
    proprio_mode:         str   = "full"
    proprio_dropout_rate: float = 0.3


def _lerobot_config(cfg):
    feats = {STATE_KEY: PolicyFeature(type=FeatureType.STATE, shape=(cfg.state_dim,))}
    norm  = {"STATE": NormalizationMode.IDENTITY, "ACTION": NormalizationMode.IDENTITY}
    if cfg.use_image:
        for _k in [f"observation.images.{c}" for c in cfg.camera_names]:
            feats[_k] = PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224))
        norm["VISUAL"] = NormalizationMode.IDENTITY
    return _LRConfig(
        n_obs_steps=1, chunk_size=cfg.chunk_size, n_action_steps=cfg.chunk_size,
        input_features=feats,
        output_features={ACTION_KEY: PolicyFeature(type=FeatureType.ACTION,
                                                   shape=(cfg.action_dim,))},
        normalization_mapping=norm,
        paligemma_variant=cfg.paligemma_variant,
        action_expert_variant=cfg.action_expert_variant,
        max_state_dim=cfg.max_state_dim, max_action_dim=cfg.max_action_dim,
        num_inference_steps=cfg.num_inference_steps,
        tokenizer_max_length=cfg.tokenizer_max_length,
        dtype=cfg.dtype, gradient_checkpointing=cfg.gradient_checkpointing,
        freeze_vision_encoder=cfg.freeze_vision_encoder,
        # with LoRA the base VLM must be frozen — adapters carry the VLM update
        train_expert_only=cfg.train_expert_only or cfg.vlm_lora_rank > 0)


def _load_pretrained_weights(policy, repo_id):
    """Load openpi-ported weights with version-proof key remapping + a HARD check.

    lerobot's own from_pretrained loads with strict=False and only prints missing
    keys — under transformers >= 5.4 (which dropped the `.vision_model` nesting
    inside SigLIP) that silently leaves the ENTIRE vision tower random-init."""
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    sd = load_file(hf_hub_download(repo_id, "model.safetensors"))
    sd = policy._fix_pytorch_state_dict_keys(sd, policy.config)
    sd = {(k if k.startswith("model.") else f"model.{k}"): v for k, v in sd.items()}
    model_keys = set(policy.state_dict().keys())
    if (any(".vision_tower.vision_model." in k for k in sd)
            and not any(".vision_tower.vision_model." in k for k in model_keys)):
        sd = {k.replace(".vision_tower.vision_model.", ".vision_tower."): v
              for k, v in sd.items()}
    missing, unexpected = policy.load_state_dict(sd, strict=False)
    n_loaded = len(model_keys) - len(missing)
    print(f"pretrained load: {n_loaded}/{len(model_keys)} tensors from {repo_id} "
          f"({len(unexpected)} unexpected ignored)")
    if n_loaded < 0.99 * len(model_keys):
        raise RuntimeError(
            f"only {n_loaded}/{len(model_keys)} tensors matched {repo_id} — a partial "
            f"load silently finetunes random weights. First missing: {sorted(missing)[:5]}")


def _inject_vlm_lora(policy, rank, alpha, dropout):
    """LoRA adapters on the (frozen) VLM's attention projections, in place —
    peft's inject_adapter_in_model keeps lerobot's module paths intact.
    Adapter params stay fp32 for stable AdamW on bf16 base weights."""
    from peft import LoraConfig, inject_adapter_in_model

    inject_adapter_in_model(
        LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=dropout,
                   target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none"),
        policy.model.paligemma_with_expert.paligemma)
    n_lora = 0
    for n, p in policy.named_parameters():
        if "lora_" in n:
            p.data = p.data.float(); p.requires_grad_(True); n_lora += p.numel()
    n_train = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    print(f"LoRA r={rank} on VLM q/k/v/o: {n_lora/1e6:.1f}M adapter params; "
          f"total trainable {n_train/1e6:.0f}M (frozen base VLM + full action expert)")


class PiPolicy(nn.Module):
    """π0.5 wrapper — mean-std norm in the wrapper, IDENTITY inside lerobot."""

    def __init__(self, cfg: PolicyCfg, stats: dict):
        super().__init__()
        self.cfg = cfg
        self.image_keys = [f"observation.images.{c}" for c in cfg.camera_names]
        self.policy = PI05Policy(_lerobot_config(cfg))
        if cfg.pretrained:
            _load_pretrained_weights(self.policy, cfg.pretrained)
        else:
            print("WARNING: random-init weights — smoke tests only, NOT finetuning")
        if cfg.vlm_lora_rank > 0:
            _inject_vlm_lora(self.policy, cfg.vlm_lora_rank,
                             cfg.vlm_lora_alpha, cfg.vlm_lora_dropout)
        self.tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
        for k in ("state_mean", "state_std", "action_mean", "action_std"):
            self.register_buffer(k, torch.as_tensor(stats[k]).float())
        self.register_buffer("_imagenet_mean", _IMN_MEAN.clone())
        self.register_buffer("_imagenet_std",  _IMN_STD.clone())

    def _norm_state(self, s):    return (s - self.state_mean) / self.state_std
    def _norm_action(self, a):   return (a - self.action_mean) / self.action_std
    def _unnorm_action(self, a): return a * self.action_std + self.action_mean
    def _to_raw(self, img):      # undo ImageNet norm -> [0,1]; lerobot maps to [-1,1]
        return (img * self._imagenet_std + self._imagenet_mean).clamp(0, 1)

    def _make_batch(self, obs_state, actions=None, action_is_pad=None,
                    obs_image=None, task=None, training=None):
        if training is None:
            training = self.training
        B = obs_state.shape[0]
        task = task or [TASK_TEXT] * B
        state = mask_state(self._norm_state(obs_state), self.cfg.proprio_mode,
                           self.cfg.proprio_dropout_rate, training)
        batch = {STATE_KEY: state}                     # (B, state_dim) — no seq dim
        if self.cfg.use_image and obs_image is not None:
            if torch.is_tensor(obs_image):                 # 1 cam -> dict
                obs_image = {self.image_keys[0]: obs_image}
            for _k in self.image_keys:                     # feed each camera
                batch[_k] = self._to_raw(obs_image[_k])
        enc = self.tokenizer(list(task), return_tensors="pt", padding="max_length",
                             truncation=True, max_length=self.cfg.tokenizer_max_length)
        batch[LANG_TOKENS]    = enc["input_ids"].to(obs_state.device)
        batch[LANG_ATTN_MASK] = enc["attention_mask"].to(obs_state.device)
        if actions is not None:
            batch[ACTION_KEY]      = self._norm_action(actions)
            batch["action_is_pad"] = action_is_pad
        return batch

    def forward(self, obs_state, actions, action_is_pad, obs_image=None, task=None):
        loss, _ = self.policy.forward(
            self._make_batch(obs_state, actions, action_is_pad, obs_image, task))
        return loss, loss.item(), 0.0

    def reset(self):
        self.policy.reset()

    @torch.no_grad()
    def predict(self, obs_state, obs_image=None, task=None):
        return self._unnorm_action(self.policy.select_action(
            self._make_batch(obs_state, obs_image=obs_image, task=task, training=False)))


def build_model(cfg: dict, stats: dict, device):
    m, d = cfg["model"], cfg["dataset"]
    return PiPolicy(PolicyCfg(
        state_dim=m["state_dim"], action_dim=m["action_dim"],
        chunk_size=d["chunk_size"], use_image=d["use_image"],
        camera_names=tuple(d.get("camera_names", ("wrist_cam", "scene_cam"))),
        tokenizer_max_length=m["tokenizer_max_length"],
        pretrained=m.get("pretrained", ""),
        vlm_lora_rank=m.get("vlm_lora_rank", 0),
        vlm_lora_alpha=m.get("vlm_lora_alpha", 32),
        vlm_lora_dropout=m.get("vlm_lora_dropout", 0.05),
        dtype=m["dtype"], gradient_checkpointing=m["gradient_checkpointing"],
        freeze_vision_encoder=m["freeze_vision_encoder"],
        train_expert_only=m["train_expert_only"],
        proprio_mode=m["proprio_mode"],
        proprio_dropout_rate=m["proprio_dropout_rate"]), stats).to(device)

print("π0.5 wrapper defined")

## 8 · Assemble the config

Mirrors the repo's `policies/pi05/config.yaml` schema — this dict is stored inside
every checkpoint, which is what lets the repo's `deploy.py` rebuild the model on the robot box.

In [ ]:
CFG = {
    "dataset": {"root": DATA_DIR, "chunk_size": CHUNK_SIZE, "use_image": True,
                "image_size": [224, 224], "val_frac": VAL_FRAC, "aug_level": AUG_LEVEL,
                "camera_names": ["wrist_cam", "scene_cam"]},
    "model": {"state_dim": STATE_DIM, "action_dim": ACTION_DIM,
              "paligemma_variant": "gemma_2b", "action_expert_variant": "gemma_300m",
              "max_state_dim": 32, "max_action_dim": 32,
              "tokenizer_max_length": 200, "num_inference_steps": 10,
              "pretrained": PRETRAINED,
              "vlm_lora_rank": LORA_RANK if FINETUNE_MODE == "lora" else 0,
              "vlm_lora_alpha": LORA_ALPHA, "vlm_lora_dropout": LORA_DROPOUT,
              "dtype": "bfloat16", "gradient_checkpointing": True,
              "freeze_vision_encoder": FINETUNE_MODE == "freeze_vision",
              "train_expert_only":     FINETUNE_MODE == "expert_only",   # lora mode freezes the VLM via vlm_lora_rank
              "proprio_mode": PROPRIO_MODE, "proprio_dropout_rate": 0.3},
    "training": {"batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
                 "max_epochs": MAX_EPOCHS, "grad_clip": GRAD_CLIP,
                 "save_every": SAVE_EVERY, "checkpoint_dir": CKPT_DIR,
                 "device": "cuda", "seed": SEED},
}
import json
print(json.dumps({"finetune": {k: CFG["model"][k] for k in
      ("pretrained", "vlm_lora_rank", "dtype", "gradient_checkpointing",
       "freeze_vision_encoder", "train_expert_only")},
      "training": CFG["training"]}, indent=2))

## 9 · Build datasets + model, time one step

Downloads the pretrained π0.5 weights (~6 GB, first run only) and confirms they load. Prints trainable/frozen parameter counts
and times one forward+backward — if this OOMs, fix it **now** (cell 1: `expert_only`
/ smaller batch, then re-run cells 1 → 4 → 8 → 9), not 20 minutes into training.
The model built here is reused by the training loop.

In [ ]:
import time, torch
from torch.utils.data import DataLoader

torch.manual_seed(SEED)

n_eps = json.loads((pathlib.Path(DATA_DIR) / "meta/info.json").read_text())["total_episodes"]
train_eps, val_eps = FR5Dataset.episode_split(n_eps, VAL_FRAC, SEED)
train_ds = FR5Dataset(DATA_DIR, CHUNK_SIZE, episode_indices=train_eps, aug_level=AUG_LEVEL)
val_ds   = FR5Dataset(DATA_DIR, CHUNK_SIZE, episode_indices=val_eps,   aug_level="none")
stats    = train_ds.get_stats()
print(f"train: {len(train_eps)} episodes / {len(train_ds)} samples   "
      f"val: {len(val_eps)} episodes / {len(val_ds)} samples")

model = build_model(CFG, stats, torch.device("cuda"))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable {trainable/1e6:.0f}M   frozen {frozen/1e6:.0f}M   ({FINETUNE_MODE})")

b = torch.utils.data.default_collate([train_ds[i] for i in range(BATCH_SIZE)])
t0 = time.time()
loss, _, _ = model(b["observation.state"].cuda(), b["action"].cuda(),
                   b["action_is_pad"].cuda(),
                   {k: b[k].cuda() for k in b if k.startswith("observation.images.")},
                   task=list(b["task"]))
loss.backward(); torch.cuda.synchronize()
step_s = time.time() - t0
spe = max(1, len(train_ds) // BATCH_SIZE)
print(f"1 step = {step_s:.2f}s  →  ~{step_s*spe/60:.1f} min/epoch, "
      f"~{step_s*spe*MAX_EPOCHS/3600:.1f} h for {MAX_EPOCHS} epochs")
print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} / {vram:.0f} GB")
model.zero_grad(set_to_none=True)

## 10 · Train

Same loop semantics as the repo's `common/train.py`: AdamW over `requires_grad` params,
grad-norm clipping, per-epoch validation, `metrics.csv`, periodic + `best.pt` checkpoints
in the **repo's checkpoint format** (so `deploy.py --checkpoint best.pt` just works).

> If the browser disconnects, the Jupyter **kernel keeps running server-side** on RunPod —
> reopen the notebook and check progress via cell 11 (outputs of this cell are lost, the
> run is not).

In [ ]:
import csv, time, pathlib, torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ckpt_dir = pathlib.Path(CKPT_DIR); ckpt_dir.mkdir(parents=True, exist_ok=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                              lr=LR, weight_decay=WEIGHT_DECAY)

metrics_path = ckpt_dir / "metrics.csv"
if not metrics_path.exists():
    with open(metrics_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_l1", "val_l1", "train_val_gap",
                                "grad_norm", "lr", "seconds"])

def run_epoch(loader, train):
    model.train(train)
    tot, gn, n = 0.0, 0.0, 0
    with (torch.enable_grad() if train else torch.no_grad()):
        for batch in tqdm(loader, leave=False, desc="train" if train else "val"):
            loss, li, _ = model(batch["observation.state"].cuda(),
                                batch["action"].cuda(), batch["action_is_pad"].cuda(),
                                {k: batch[k].cuda() for k in batch if k.startswith("observation.images.")},
                                task=list(batch["task"]))
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                gn += torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRAD_CLIP).item()
                optimizer.step()
            tot += li; n += 1
    return tot / max(n, 1), gn / max(n, 1)

run = None
if USE_WANDB:
    run = wandb.init(project=WANDB_PROJECT,
                     name=f"{POLICY}-{FINETUNE_MODE}-bs{BATCH_SIZE}",
                     config={**CFG["model"], **CFG["training"],
                             "policy": POLICY, "finetune_mode": FINETUNE_MODE,
                             "dataset_repo": HF_DATASET_REPO,
                             "train_episodes": len(train_eps), "val_episodes": len(val_eps)})

best_val = float("inf")
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    train_l1, grad_norm = run_epoch(train_loader, True)
    val_l1, _           = run_epoch(val_loader, False)
    secs = time.time() - t0
    with open(metrics_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, f"{train_l1:.6f}", f"{val_l1:.6f}",
                                f"{val_l1 - train_l1:.6f}", f"{grad_norm:.4f}",
                                LR, f"{secs:.1f}"])
    if run:
        run.log({"train/loss": train_l1, "val/loss": val_l1,
                 "val/gap": val_l1 - train_l1, "train/grad_norm": grad_norm,
                 "train/lr": LR, "epoch_seconds": secs}, step=epoch)
    is_best = val_l1 < best_val
    best_val = min(best_val, val_l1)
    print(f"epoch {epoch:3d}/{MAX_EPOCHS}  train {train_l1:.4f}  val {val_l1:.4f}"
          f"{'  ↑ best' if is_best else ''}  ({secs:.0f}s)")
    if is_best or epoch % SAVE_EVERY == 0 or epoch == MAX_EPOCHS:
        ckpt = {"epoch": epoch, "policy": POLICY, "model_state": model.state_dict(),
                "val_l1": val_l1, "config": CFG, "stats": stats,
                "action_space": train_ds.info.get("action_space", "joint")}
        if epoch % SAVE_EVERY == 0 or epoch == MAX_EPOCHS:
            torch.save(ckpt, ckpt_dir / f"epoch_{epoch:04d}.pt")
        if is_best:
            torch.save(ckpt, ckpt_dir / "best.pt")

if run:
    run.summary["best_val_l1"] = best_val
print(f"done — best val_l1 {best_val:.4f}   checkpoints in {ckpt_dir}")

## 11 · Monitor — re-run any time (also after a browser reconnect)

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt

csv_path = pathlib.Path(CKPT_DIR, "metrics.csv")
if csv_path.exists():
    m = pd.read_csv(csv_path)
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
    m.plot(x="epoch", y=["train_l1", "val_l1"], ax=ax[0], title="flow-matching loss")
    m.plot(x="epoch", y="train_val_gap", ax=ax[1], title="train/val gap (overfitting watch)")
    m.plot(x="epoch", y="grad_norm", ax=ax[2], title="grad norm")
    m.plot(x="epoch", y="seconds", ax=ax[3], title="seconds / epoch")
    plt.tight_layout(); plt.show()
    best = m.loc[m.val_l1.idxmin()]
    print(f"best val_l1 = {best.val_l1:.4f} @ epoch {int(best.epoch)}  ({len(m)} epochs logged)")
else:
    print("metrics.csv not written yet — appears after epoch 1")

## 12 · Ground-truth vs prediction — the real quality check

Rolls the trained policy over one **validation** episode (open-loop chunks, like the
repo's eval layer) and overlays predicted joint trajectories on the ground truth.
Numbers lie less than loss curves: look for tracking through the grasp region.
Logged to wandb as a figure when enabled.

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval(); model.reset()
gt, pred, last_ep = [], [], None
with torch.no_grad():
    for i in range(len(val_ds)):
        ep_idx = val_ds._samples[i][0]
        if ep_idx != last_ep:          # new episode -> clear the action-chunk queue
            model.reset(); last_ep = ep_idx
        item = val_ds[i]
        gt.append(item["action"][0])
        a = model.predict(item["observation.state"].unsqueeze(0).cuda(),
                          obs_image={k: item[k].unsqueeze(0).cuda() for k in item if k.startswith("observation.images.")},
                          task=[item["task"]])
        pred.append(a.squeeze(0).float().cpu())
gt, pred = torch.stack(gt), torch.stack(pred)

D = gt.shape[1]                                   # action dim (auto, not hardcoded)
names = ([f"joint{j+1}" for j in range(6)] + ["gripper"]) if D == 7 else [f"a{j}" for j in range(D)]
mae = (gt - pred).abs().mean(0)
ncols = 4
nrows = -(-(D + 1) // ncols)                      # ceil: one panel per dim + MAE summary
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()
for j in range(D):
    axes[j].plot(gt[:, j], label="ground truth", lw=1.5)
    axes[j].plot(pred[:, j], label="prediction", lw=1.0, alpha=0.85)
    axes[j].set_title(names[j]); axes[j].legend(fontsize=7)
axes[D].axis("off")
axes[D].text(0, 0.5, "MAE per dim:\n" +
             "\n".join(f"{n}: {v:.3f}" for n, v in zip(names, mae)), fontsize=9)
for k in range(D + 1, len(axes)):
    axes[k].axis("off")
plt.suptitle(f"{POLICY} — GT vs prediction, validation episodes ({len(val_ds)} steps)")
plt.tight_layout(); plt.show()
print("mean MAE:", f"{mae.mean():.4f}")

if USE_WANDB and wandb.run:
    wandb.log({"eval/gt_vs_pred": wandb.Image(fig), "eval/mae_mean": mae.mean().item()})
    wandb.finish()

## 13 · Ship checkpoints to the Hub

Uploads `best.pt` + `metrics.csv` to a private model repo so they survive pod termination.
On the robot box: download `best.pt` and run `python common/deploy.py --checkpoint best.pt`.

In [ ]:
from huggingface_hub import HfApi, whoami

repo = f"{whoami()['name']}/fr5-{POLICY}-{FINETUNE_MODE}"
api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=CKPT_DIR, repo_id=repo,
                  allow_patterns=["best.pt", "metrics.csv"],
                  commit_message=f"{POLICY} {FINETUNE_MODE} bs={BATCH_SIZE} epochs={MAX_EPOCHS}")
print(f"uploaded -> https://huggingface.co/{repo}")

## Troubleshooting

**CUDA OOM** — in order of preference:
1. `FINETUNE_MODE = "expert_only"` (freezes the 2B VLM; the 300M expert still learns the task)
2. halve `BATCH_SIZE` (set it explicitly, e.g. `BATCH_SIZE = 1`)
3. both

Both live in **cell 1 (Parameters)** — edit there, restart the kernel (frees VRAM cleanly),
then run cells **1 → 9** again and relaunch cell 10.

**`GatedRepoError` / 403 on PaliGemma** — the HF account hasn't accepted the license, or the token lacks read scope.

**Pod / kernel restarted mid-run** — checkpoints land every `SAVE_EVERY` (10) epochs plus `best.pt`.
There's no optimizer-state resume; a relaunch of cell 10 restarts from scratch weights, so for
spot pods prefer shorter `MAX_EPOCHS` per run and upload checkpoints (cell 12) as you go.

**Throughput sanity** — cell 9 prints measured step time, min/epoch, full-run hours, and peak VRAM.
Watch the first epochs for a *decreasing* train loss before walking away.